# Feature selection — wsBNN (weight-sharing Bayesian Neural Network)

Embedded method theo Mishra et al., *BMC Bioinformatics* 27:79 (2026). Lớp đầu dùng spike-and-slab prior với **một biến bao gồm z_i dùng chung** cho mọi trọng số xuất phát từ gene i; các lớp sau Gaussian mean-field. Suy diễn variational Bayes (loss = −ELBO = NLL + KL), z_i nới lỏng bằng Gumbel-sigmoid. Gene được xếp hạng theo xác suất bao gồm hậu nghiệm φ_i = sigmoid(θ_i), lấy top `N_FEATURES` (bài báo lấy top 10/30 theo φ). Siêu tham số mặc định lấy theo code gốc (github.com/AkankshaMishra/wsBNN, `classification/brca`): hidden (16, 8, 4), τ = 0.3, σ_prior = 1.3, φ_prior tính theo công thức trong code, ρ0 = −6, φ khởi tạo 0.99, μ ~ U(−0.6, 0.6), 30 mẫu Monte Carlo/bước, Adam lr = 1e-3, full-batch, 50 000 epoch.**Tự dùng GPU nếu có** (Kaggle: Settings → Accelerator → GPU). Mặc định `PREPROCESS = "none"` (giữ log2 gốc, theo quy ước của đề tài). Lưu ý bài báo gốc có chuẩn hoá: code wsBNN chia mỗi gene cho chuẩn L2 (`preprocessing.normalize(X, axis=0)`). Có thể thử `PREPROCESS = "l2_col"` để giống bài báo — chỉ áp bên trong bước chọn gene, dữ liệu gốc không đổi.

- **Đầu vào:** `DATA_DIR/fold_{1..5}/train.h5` (chỉ dùng tập **train**; val/test không đụng tới). Dữ liệu giữ nguyên log2 expression, không chuẩn hoá.
- **Đổi dataset:** sửa `DATA_DIR` ở cell cấu hình. Số gene chọn: `N_FEATURES` (mặc định 100).
- **Đầu ra** (`/kaggle/working/feature_selection/wsbnn/<dataset>/`):
  - `selected_genes/fold_k.csv`, `selected_genes_all_folds.csv`, `selected_genes.json` — gene được chọn theo fold (rank, score)
  - `metrics_per_fold.csv`, `metrics_summary.csv` — thời gian, CPU utilization, peak memory, energy, EDP, carbon footprint
- **Đo lường:** chỉ bao quanh lời gọi `select_features` (không tính thời gian đọc file). Energy/Carbon ước lượng bằng `codecarbon` (tracking_mode="process"); EDP = Energy(J) × Time(s).

In [ ]:
# tables: backend cho pd.read_hdf | codecarbon: do nang luong/CO2 | psutil: CPU & RAM
get_ipython().system("pip install -q codecarbon psutil tables")

In [ ]:
import os

# ====== CAU HINH (sua tay) ======
# Doi DATA_DIR sang bo khac (vd .../h5/tcga_luad) de chay dataset khac
DATA_DIR = "/kaggle/input/datasets/lfreedom2750/experiment-dataset/h5/gse68465"
N_FEATURES = 100          # so gene giu lai moi fold
N_FOLDS = 5
SEED = 42
METHOD = "wsbnn"
CLASS_WEIGHT = "balanced"      # None | "balanced": trong so lop trong cross-entropy
PREPROCESS = "none"            # "none" (log2 goc) | "l2_col" | "zscore" -- chi ben trong buoc chon gene
WSBNN_HIDDEN = (16, 8, 4)
WSBNN_TEMP = 0.3              # nhiet do Gumbel (code goc 0.3; bai bao thu 0.1-0.9, tot nhat 0.7 tren du lieu mo phong)
WSBNN_PHI_PRIOR = "auto"      # "auto" = cong thuc trong code goc, hoac so trong (0, 1)
WSBNN_SIGMA_PRIOR = 1.3       # do lech chuan prior slab / cac lop sau (softplus(rho_prior) trong code goc)
WSBNN_INIT_INCLUSION = 0.99   # xac suat bao gom khoi tao
WSBNN_RHO0 = -6.0             # khoi tao rho -> sigma = softplus(rho0)
WSBNN_LR = 1e-3
WSBNN_EPOCHS = 50000          # full-batch; giam neu chay CPU
WSBNN_MC_SAMPLES = 30         # so mau Monte Carlo moi buoc (code goc: sample_elbo(..., 30, ...))
WSBNN_LOG_EVERY = 5000
METHOD_PARAMS = {"hidden": WSBNN_HIDDEN, "temperature": WSBNN_TEMP, "phi_prior": WSBNN_PHI_PRIOR,
                 "sigma_prior": WSBNN_SIGMA_PRIOR, "init_inclusion": WSBNN_INIT_INCLUSION, "rho0": WSBNN_RHO0,
                 "lr": WSBNN_LR, "epochs": WSBNN_EPOCHS, "mc_samples": WSBNN_MC_SAMPLES, "class_weight": CLASS_WEIGHT, "preprocess": PREPROCESS}
# None -> codecarbon tu dinh vi (can Internet); hoac dat ma ISO 3 ky tu, vd "USA", "VNM"
COUNTRY_ISO_CODE = None

DATASET_NAME = os.path.basename(os.path.normpath(DATA_DIR))
RESULT_DIR = os.path.join("/kaggle/working/feature_selection", METHOD, DATASET_NAME)
os.makedirs(os.path.join(RESULT_DIR, "selected_genes"), exist_ok=True)

assert os.path.isfile(os.path.join(DATA_DIR, "fold_1", "train.h5")), \
    f"Khong thay {DATA_DIR}/fold_1/train.h5 -- da Add Input dataset chua?"
print("Dataset :", DATASET_NAME, "->", DATA_DIR)
print("Method  :", METHOD, "| N_FEATURES =", N_FEATURES)
print("Ket qua :", RESULT_DIR)
print("So CPU  :", os.cpu_count())

## Đọc dữ liệu

In [ ]:
import json, time, threading, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

STAGE_NAMES = {0: "Normal", 1: "I", 2: "II", 3: "III", 4: "IV"}


def load_split(fold, split="train"):
    # File h5 (pandas, key="data"): index = sample_id, cot = ten gene + cot "label"
    df = pd.read_hdf(os.path.join(DATA_DIR, f"fold_{fold}", f"{split}.h5"), key="data")
    X = df.drop(columns="label")
    # Giu nguyen gia tri log2 expression, KHONG chuan hoa
    return X.values.astype(np.float64), df["label"].values.astype(int), X.columns.to_numpy(), df.index.to_numpy()


# Doc thu fold 1 de xem cau truc
X0, y0, genes0, ids0 = load_split(1)
print(f"fold_1/train: {X0.shape[0]} mau x {X0.shape[1]} gene | khoang gia tri [{X0.min():.2f}, {X0.max():.2f}]")
print("Vi du gene:", list(genes0[:5]), "| vi du sample:", list(ids0[:3]))
print("Phan bo lop:", pd.Series(y0).map(STAGE_NAMES).value_counts().to_dict())
del X0, y0, genes0, ids0

## Bộ đo tài nguyên

In [ ]:
import psutil

try:
    from codecarbon import EmissionsTracker, OfflineEmissionsTracker
    _CODECARBON = True
except ImportError:
    _CODECARBON = False


class ResourceTracker:
    """Do tren 1 khoi lenh: thoi gian (s), CPU utilization, peak memory (RSS, gom ca
    tien trinh con cua joblib), energy (kWh, codecarbon), carbon (kg CO2eq) va
    Energy-Delay Product EDP = Energy(J) * Time(s)."""

    def __init__(self, name, interval=0.05):
        self.name, self.interval = name, interval
        self.proc = psutil.Process()
        self._stop = threading.Event()

    def _procs(self):
        try:
            return [self.proc] + self.proc.children(recursive=True)
        except psutil.Error:
            return [self.proc]

    def _poll(self):
        # RSS tong va CPU time cua worker con (tru phan CPU worker da dung truoc khi bat dau do)
        rss = 0
        for p in self._procs():
            try:
                rss += p.memory_info().rss
                if p.pid != self.proc.pid:
                    t = p.cpu_times()
                    self._child_cpu[p.pid] = t.user + t.system - self._child_base.get(p.pid, 0.0)
            except psutil.Error:
                pass
        self._peak_rss = max(self._peak_rss, rss)
        return rss

    def _sampler(self):
        while not self._stop.is_set():
            self._poll()
            self._sys_cpu.append(psutil.cpu_percent(interval=None))
            self._stop.wait(self.interval)

    def __enter__(self):
        self._child_cpu, self._sys_cpu, self._peak_rss = {}, [], 0
        self._child_base = {}
        for p in self._procs()[1:]:
            try:
                t = p.cpu_times()
                self._child_base[p.pid] = t.user + t.system
            except psutil.Error:
                pass
        self.tracker = None
        if _CODECARBON:
            try:
                kw = dict(project_name=self.name, output_dir=RESULT_DIR, save_to_file=False,
                          log_level="error", measure_power_secs=1, tracking_mode="process")
                self.tracker = (OfflineEmissionsTracker(country_iso_code=COUNTRY_ISO_CODE, **kw)
                                if COUNTRY_ISO_CODE else EmissionsTracker(**kw))
                self.tracker.start()
            except Exception as e:
                print("[ResourceTracker] codecarbon loi:", e)
                self.tracker = None
        self.base_rss = self._poll()
        self._child_cpu = {}
        psutil.cpu_percent(interval=None)
        t = self.proc.cpu_times()
        self._cpu0 = t.user + t.system
        self._stop.clear()
        self._thread = threading.Thread(target=self._sampler, daemon=True)
        self._thread.start()
        self._t0 = time.perf_counter()
        return self

    def __exit__(self, *exc):
        self.time_sec = time.perf_counter() - self._t0
        t = self.proc.cpu_times()
        main_cpu = t.user + t.system - self._cpu0
        self._stop.set()
        self._thread.join(timeout=2)
        self._poll()
        cpu_sec = main_cpu + sum(self._child_cpu.values())

        self.energy_kwh, self.carbon_kg = float("nan"), float("nan")
        if self.tracker is not None:
            try:
                self.carbon_kg = float(self.tracker.stop() or 0.0)
                data = self.tracker.final_emissions_data
                self.energy_kwh = float(data.energy_consumed) if data is not None else float("nan")
            except Exception as e:
                print("[ResourceTracker] codecarbon stop loi:", e)

        n_cpu = os.cpu_count() or 1
        self.metrics = {
            "time_sec": self.time_sec,
            "cpu_time_sec": cpu_sec,
            # % tren tong nang luc may (100% = dung het n_cpu core)
            "cpu_util_percent": 100.0 * cpu_sec / (self.time_sec * n_cpu) if self.time_sec > 0 else float("nan"),
            "avg_cores_used": cpu_sec / self.time_sec if self.time_sec > 0 else float("nan"),
            "system_cpu_percent_mean": float(np.mean(self._sys_cpu)) if self._sys_cpu else float("nan"),
            "peak_memory_mb": self._peak_rss / 1024 ** 2,
            "peak_memory_increase_mb": (self._peak_rss - self.base_rss) / 1024 ** 2,
            "energy_kwh": self.energy_kwh,
            "energy_joule": self.energy_kwh * 3.6e6,
            "edp_joule_sec": self.energy_kwh * 3.6e6 * self.time_sec,
            "carbon_kg_co2eq": self.carbon_kg,
            "carbon_g_co2eq": self.carbon_kg * 1e3,
        }
        return False

## Thuật toán: wsBNN (weight-sharing Bayesian Neural Network)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# Tu dung GPU neu co (Kaggle: Settings -> Accelerator -> GPU)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None
print("Thiet bi:", DEVICE, "|", GPU_NAME or f"{os.cpu_count()} CPU", "| torch", torch.__version__)
if DEVICE.type == "cuda":
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SELECT_INFO = {}   # thong tin phu cua lan chon gan nhat, duoc ghi vao metrics_per_fold.csv


def _preprocess(X):
    # "none": giu log2 goc | "l2_col": chia moi gene cho chuan L2 (nhu code wsBNN) | "zscore": nhu scFSNN
    if PREPROCESS == "none":
        return X
    if PREPROCESS == "l2_col":
        nrm = np.linalg.norm(X, axis=0)
        nrm[nrm == 0] = 1.0
        return X / nrm
    if PREPROCESS == "zscore":
        sd = X.std(0)
        sd[sd == 0] = 1.0
        return (X - X.mean(0)) / sd
    raise ValueError(PREPROCESS)


def _class_weight(yc, n_class):
    if CLASS_WEIGHT != "balanced":
        return None
    cnt = np.bincount(yc, minlength=n_class)
    return torch.tensor(len(yc) / (n_class * cnt), dtype=torch.float32, device=DEVICE)


def _gpu_begin():
    SELECT_INFO.clear()
    SELECT_INFO.update(device=DEVICE.type, gpu_name=GPU_NAME)
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def _gpu_end():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
        SELECT_INFO["gpu_peak_memory_mb"] = torch.cuda.max_memory_allocated() / 1024 ** 2


def _kl_gauss(mu, sigma, sigma_p):
    # KL( N(mu, sigma^2) || N(0, sigma_p^2) ) tung phan tu
    return torch.log(sigma_p / sigma) + 0.5 * (sigma ** 2 + mu ** 2) / sigma_p ** 2 - 0.5


class SharedSpikeSlabLayer(nn.Module):
    """Lop dau cua wsBNN: moi gene i co 1 bien bao gom z_i ~ Bernoulli(phi_i) DUNG CHUNG cho moi
    trong so w_i. noi gene i toi lop an thu nhat; w_ij | z_i ~ (1 - z_i) delta_0 + z_i N(mu_ij, sigma_ij^2).
    z_i duoc noi long bang Gumbel-sigmoid (hard, straight-through) voi nhiet do temp.
    Bias lop dau cung co gate spike-and-slab rieng tung node (nhu SpikeNSlabLayerModified trong code goc)."""

    def __init__(self, d, h, phi_prior, sigma_prior, init_incl, rho0):
        super().__init__()
        self.phi_prior, self.sigma_prior = phi_prior, sigma_prior
        self.w_mu = nn.Parameter(torch.empty(d, h).uniform_(-0.6, 0.6))
        self.w_rho = nn.Parameter(torch.full((d, h), rho0))
        self.b_mu = nn.Parameter(torch.empty(h).uniform_(-0.6, 0.6))
        self.b_rho = nn.Parameter(torch.full((h,), rho0))
        self.theta = nn.Parameter(torch.full((d,), math.log(init_incl / (1 - init_incl))))
        self.b_theta = nn.Parameter(torch.full((h,), math.log(init_incl / (1 - init_incl))))

    def inclusion_prob(self):
        return torch.sigmoid(self.theta)

    @staticmethod
    def _gate(theta, temp):
        u = torch.rand_like(theta).clamp(1e-6, 1 - 1e-6)
        z_soft = torch.sigmoid((theta + torch.log(u) - torch.log1p(-u)) / temp)
        return (z_soft > 0.5).float() + z_soft - z_soft.detach()     # hard=True, straight-through

    def _kl_spike_slab(self, theta, mu, sigma, sp):
        # KL giong code goc: phi duoc expand theo moi trong so cua hang i
        phi = torch.sigmoid(theta).clamp(1e-7, 1 - 1e-7)
        if mu.dim() == 2:
            phi = phi[:, None].expand_as(mu)
        kl = phi * (torch.log(phi) - math.log(self.phi_prior)) + \
            (1 - phi) * (torch.log(1 - phi) - math.log(1 - self.phi_prior)) + phi * _kl_gauss(mu, sigma, sp)
        return kl.sum()

    def forward(self, x, temp):
        z_w, z_b = self._gate(self.theta, temp), self._gate(self.b_theta, temp)
        w_sigma, b_sigma = F.softplus(self.w_rho), F.softplus(self.b_rho)
        w = (self.w_mu + w_sigma * torch.randn_like(w_sigma)) * z_w[:, None]
        b = (self.b_mu + b_sigma * torch.randn_like(b_sigma)) * z_b
        sp = torch.tensor(self.sigma_prior, device=x.device)
        self.kl = (self._kl_spike_slab(self.theta, self.w_mu, w_sigma, sp)
                   + self._kl_spike_slab(self.b_theta, self.b_mu, b_sigma, sp))
        return x @ w + b


class GaussianLayer(nn.Module):
    """Cac lop sau: trong so va bias ~ N(mu, sigma^2) mean-field, prior N(0, sigma_prior^2)."""

    def __init__(self, d, h, sigma_prior, rho0):
        super().__init__()
        self.sigma_prior = sigma_prior
        self.w_mu = nn.Parameter(torch.empty(d, h).uniform_(-0.6, 0.6))
        self.w_rho = nn.Parameter(torch.full((d, h), rho0))
        self.b_mu = nn.Parameter(torch.empty(h).uniform_(-0.6, 0.6))
        self.b_rho = nn.Parameter(torch.full((h,), rho0))

    def forward(self, x, temp=None):
        w_sigma, b_sigma = F.softplus(self.w_rho), F.softplus(self.b_rho)
        w = self.w_mu + w_sigma * torch.randn_like(w_sigma)
        b = self.b_mu + b_sigma * torch.randn_like(b_sigma)
        sp = torch.tensor(self.sigma_prior, device=x.device)
        self.kl = _kl_gauss(self.w_mu, w_sigma, sp).sum() + _kl_gauss(self.b_mu, b_sigma, sp).sum()
        return x @ w + b


class WSBNN(nn.Module):
    def __init__(self, d, n_class, n_train):
        super().__init__()
        dims = [d, *WSBNN_HIDDEN, n_class]
        phi_prior = WSBNN_PHI_PRIOR if WSBNN_PHI_PRIOR != "auto" else self.auto_phi_prior(dims, n_train)
        self.phi_prior = phi_prior
        self.first = SharedSpikeSlabLayer(d, dims[1], phi_prior, WSBNN_SIGMA_PRIOR, WSBNN_INIT_INCLUSION, WSBNN_RHO0)
        self.rest = nn.ModuleList([GaussianLayer(dims[i], dims[i + 1], WSBNN_SIGMA_PRIOR, WSBNN_RHO0)
                                   for i in range(1, len(dims) - 1)])

    @staticmethod
    def auto_phi_prior(dims, n):
        # Cong thuc lambda (phi_prior) trong code goc wsBNN (classification/brca/test_fsbnn_sm.py)
        # total giong code goc: (d+1)*16 + (16+1)*8 + (8+1)*4 + (4+1)*1 (so hang cuoi nhan 1, khong nhan so lop)
        total = sum((dims[i] + 1) * dims[i + 1] for i in range(len(dims) - 2)) + (dims[-2] + 1)
        hidden = dims[1:-1]
        a = math.log(total) + 0.1 * (math.log(hidden[0]) + 0.1 * sum(math.log(h) for h in hidden[1:])
                                     + math.log(math.sqrt(n) * dims[0]))
        return 1.0 / math.exp(a)

    def forward(self, x, temp):
        h = F.relu(self.first(x, temp))
        for i, layer in enumerate(self.rest):
            h = layer(h)
            if i < len(self.rest) - 1:
                h = F.relu(h)
        return h   # logits; softmax nam trong cross-entropy

    def kl(self):
        return self.first.kl + sum(layer.kl for layer in self.rest)


def select_features(X, y, k):
    _gpu_begin()
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    classes, yc = np.unique(y, return_inverse=True)
    Xt = torch.tensor(_preprocess(X), dtype=torch.float32, device=DEVICE)
    yt = torch.tensor(yc, dtype=torch.long, device=DEVICE)
    cw = _class_weight(yc, len(classes))

    net = WSBNN(X.shape[1], len(classes), len(yc)).to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), lr=WSBNN_LR)
    for ep in range(1, WSBNN_EPOCHS + 1):   # full-batch nhu code goc
        opt.zero_grad(set_to_none=True)
        nll = 0.0
        for _ in range(WSBNN_MC_SAMPLES):    # uoc luong Monte Carlo cua E_q[log-likelihood]
            logits = net(Xt, WSBNN_TEMP)
            nll = nll + F.cross_entropy(logits, yt, weight=cw, reduction="sum")
        nll = nll / WSBNN_MC_SAMPLES
        loss = nll + net.kl()                # -ELBO (KL giai tich, full-batch)
        loss.backward()
        opt.step()
        if ep % WSBNN_LOG_EVERY == 0 or ep == WSBNN_EPOCHS:
            with torch.no_grad():
                phi = net.first.inclusion_prob()
                acc = (logits.argmax(1) == yt).float().mean().item()
            print(f"    epoch {ep:6d} | -ELBO {loss.item():.1f} | NLL {nll.item():.2f} | "
                  f"train acc {acc:.3f} | gene phi>0.5: {int((phi > 0.5).sum())}")

    with torch.no_grad():
        phi = net.first.inclusion_prob().cpu().numpy().astype(float)
        wnorm = net.first.w_mu.norm(dim=1).cpu().numpy().astype(float)
    _gpu_end()
    # Xep hang theo xac suat bao gom hau nghiem phi_i; hoa (phi bao hoa ~1) -> tach bang ||mu_i.||
    order = np.lexsort((-wnorm, -phi))[:k]
    SELECT_INFO.update(phi_prior=net.phi_prior, n_phi_gt_05=int((phi > 0.5).sum()),
                       phi_at_k=float(phi[order[-1]]), final_neg_elbo=float(loss.item()))
    return order, phi[order]

## Chạy trên 5 fold

In [ ]:
all_rows, metric_rows = [], []

for fold in range(1, N_FOLDS + 1):
    X, y, genes, _ = load_split(fold, "train")   # CHI dung tap train cho feature selection
    k = min(N_FEATURES, X.shape[1])

    with ResourceTracker(f"{METHOD}_{DATASET_NAME}_fold{fold}") as rt:
        idx, scores = select_features(X, y, k)

    sel = pd.DataFrame({"fold": fold, "rank": np.arange(1, len(idx) + 1),
                        "gene": genes[idx], "score": scores})
    sel.to_csv(os.path.join(RESULT_DIR, "selected_genes", f"fold_{fold}.csv"), index=False)
    all_rows.append(sel)

    m = {"dataset": DATASET_NAME, "method": METHOD, "fold": fold,
         "n_train": X.shape[0], "n_genes_total": X.shape[1], "n_selected": len(idx),
         **SELECT_INFO, **rt.metrics}
    metric_rows.append(m)
    print(f"fold {fold}: {X.shape} -> {len(idx)} gene | {m['time_sec']:.2f}s | CPU {m['cpu_util_percent']:.1f}% "
          f"({m['avg_cores_used']:.2f} core) | peak {m['peak_memory_mb']:.0f} MB | "
          f"{m['energy_kwh']:.3e} kWh | {m['carbon_g_co2eq']:.4f} gCO2eq | top5: {list(sel.gene[:5])}")
    del X, y

selected = pd.concat(all_rows, ignore_index=True)
selected.to_csv(os.path.join(RESULT_DIR, "selected_genes_all_folds.csv"), index=False)
with open(os.path.join(RESULT_DIR, "selected_genes.json"), "w") as f:
    json.dump({f"fold_{r}": g.gene.tolist() for r, g in selected.groupby("fold")}, f, indent=1)

metrics = pd.DataFrame(metric_rows)
metrics.to_csv(os.path.join(RESULT_DIR, "metrics_per_fold.csv"), index=False)
metrics

## Tổng hợp độ đo

In [ ]:
cols = ["time_sec", "cpu_time_sec", "cpu_util_percent", "avg_cores_used", "system_cpu_percent_mean",
        "peak_memory_mb", "peak_memory_increase_mb", "energy_kwh", "energy_joule",
        "edp_joule_sec", "carbon_kg_co2eq", "carbon_g_co2eq"]
summary = metrics[cols].agg(["mean", "std", "min", "max"]).T
summary.loc["time_sec", "total"] = metrics["time_sec"].sum()
summary.loc["energy_kwh", "total"] = metrics["energy_kwh"].sum()
summary.loc["carbon_kg_co2eq", "total"] = metrics["carbon_kg_co2eq"].sum()
summary.to_csv(os.path.join(RESULT_DIR, "metrics_summary.csv"))

with open(os.path.join(RESULT_DIR, "config.json"), "w") as f:
    json.dump({"dataset": DATASET_NAME, "data_dir": DATA_DIR, "method": METHOD, "n_features": N_FEATURES,
               "n_folds": N_FOLDS, "seed": SEED, "params": METHOD_PARAMS, "cpu_count": os.cpu_count(),
               "country_iso_code": COUNTRY_ISO_CODE}, f, indent=1)
summary

## Độ ổn định tập gene giữa các fold (tham khảo)

In [ ]:
# Tham khao nhanh: do on dinh tap gene giua cac fold (Jaccard cap doi, trung binh)
sets = {r: set(g.gene) for r, g in selected.groupby("fold")}
folds = sorted(sets)
jac = pd.DataFrame(index=folds, columns=folds, dtype=float)
for a in folds:
    for b in folds:
        jac.loc[a, b] = len(sets[a] & sets[b]) / len(sets[a] | sets[b])
pairs = [jac.loc[a, b] for i, a in enumerate(folds) for b in folds[i + 1:]]
print(f"Jaccard trung binh giua cac fold: {np.mean(pairs):.3f}")
freq = selected.gene.value_counts()
print(f"Gene xuat hien o ca {N_FOLDS} fold: {int((freq == N_FOLDS).sum())}")
freq.rename("n_folds").to_csv(os.path.join(RESULT_DIR, "gene_frequency.csv"))
jac.round(3)

In [ ]:
import shutil
zip_path = shutil.make_archive(f"/kaggle/working/fs_{METHOD}_{DATASET_NAME}", "zip", RESULT_DIR)
print("Da nen:", zip_path)
for root, _, files in os.walk(RESULT_DIR):
    for fn in sorted(files):
        print(" ", os.path.relpath(os.path.join(root, fn), RESULT_DIR))